In [5]:
import pandas as pd
df = pd.read_csv("preprocessed_ipl_ball_by_ball.csv")

print(df.shape)
df.head()


(57005, 73)


/tmp/ipython-input-1082837525.py:2: DtypeWarning: Columns (42,50,62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("preprocessed_ipl_ball_by_ball.csv")


,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over_bat,ball,ball_no,...,runs,balls,fours,sixes,sr,over_bowl,maidens,runs_conceded,wickets,dots
0,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,0.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0.2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,0.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
batting_match = df.groupby(
    ["match_id", "date", "batter", "batting_team", "bowling_team", "venue"]
).agg(
    runs=("runs_batter", "sum"),
    balls=("valid_ball", "sum"),
    fours=("runs_batter", lambda x: (x == 4).sum()),
    sixes=("runs_batter", lambda x: (x == 6).sum())
).reset_index()

print("Batting Match Shape:", batting_match.shape)
batting_match.head()


Batting Match Shape: (3628, 10)


,match_id,date,batter,batting_team,bowling_team,venue,runs,balls,fours,sixes
0,335982,2008-04-18,AA Noffke,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,9,10,1,0
1,335982,2008-04-18,B Akhil,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0,2,0,0
2,335982,2008-04-18,BB McCullum,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,158,73,10,13
3,335982,2008-04-18,CL White,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,6,10,0,0
4,335982,2008-04-18,DJ Hussey,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,12,12,1,0


In [7]:
bowling_match = df.groupby(
    ["match_id", "date", "bowler", "bowling_team", "batting_team", "venue"]
).agg(
    wickets=("bowler_wicket", "sum"),
    runs_conceded=("runs_bowler", "sum"),
    balls_bowled=("valid_ball", "sum")
).reset_index()

bowling_match["overs"] = bowling_match["balls_bowled"] / 6
print("Bowling Match Shape:", bowling_match.shape)
bowling_match.head()

Bowling Match Shape: (2918, 10)


,match_id,date,bowler,bowling_team,batting_team,venue,wickets,runs_conceded,balls_bowled,overs
0,335982,2008-04-18,AA Noffke,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1.0,40,24,4.0
1,335982,2008-04-18,AB Agarkar,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,3.0,25,24,4.0
2,335982,2008-04-18,AB Dinda,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,2.0,9,18,3.0
3,335982,2008-04-18,CL White,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,0.0,24,6,1.0
4,335982,2008-04-18,I Sharma,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,1.0,7,18,3.0


In [8]:
batting_match["form_last_5"] = (
    batting_match
    .groupby("batter")["runs"]
    .rolling(5)
    .mean()
    .reset_index(level=0, drop=True)
)

batting_match[["batter", "runs", "form_last_5"]].head(10)


,batter,runs,form_last_5
0,AA Noffke,9,NaN
1,B Akhil,0,NaN
2,BB McCullum,158,NaN
3,CL White,6,NaN
4,DJ Hussey,12,NaN
5,JH Kallis,8,NaN
6,MV Boucher,7,NaN
7,Mohammad Hafeez,5,NaN
8,P Kumar,18,NaN
9,R Dravid,2,NaN


In [9]:
bowling_match["wkts_last_5"] = (
    bowling_match.groupby("bowler")["wickets"]
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
bowling_match[["bowler", "wickets", "wkts_last_5"]].head(10)

,bowler,wickets,wkts_last_5
0,AA Noffke,1.0,1.0
1,AB Agarkar,3.0,3.0
2,AB Dinda,2.0,2.0
3,CL White,0.0,0.0
4,I Sharma,1.0,1.0
5,JH Kallis,1.0,1.0
6,LR Shukla,1.0,1.0
7,P Kumar,0.0,0.0
8,SB Joshi,0.0,0.0
9,SC Ganguly,2.0,2.0


In [10]:
batting_match["venue_avg"] = (
    batting_match
    .groupby(["batter", "venue"])["runs"]
    .transform("mean")
)

batting_match[["batter", "venue", "venue_avg"]].head()


,batter,venue,venue_avg
0,AA Noffke,M Chinnaswamy Stadium,9.0
1,B Akhil,M Chinnaswamy Stadium,2.5
2,BB McCullum,M Chinnaswamy Stadium,75.0
3,CL White,M Chinnaswamy Stadium,9.2
4,DJ Hussey,M Chinnaswamy Stadium,12.0


In [11]:
bowling_match["venue_avg_wkts"] = (
    bowling_match.groupby(["bowler", "venue"])["wickets"]
    .transform("mean")
)

bowling_match[["bowler", "venue", "venue_avg_wkts"]].head()

,bowler,venue,venue_avg_wkts
0,AA Noffke,M Chinnaswamy Stadium,1.0
1,AB Agarkar,M Chinnaswamy Stadium,1.5
2,AB Dinda,M Chinnaswamy Stadium,2.0
3,CL White,M Chinnaswamy Stadium,0.0
4,I Sharma,M Chinnaswamy Stadium,0.5


In [12]:
batting_match["opponent_avg_run"] = (
    batting_match
    .groupby(["batter", "bowling_team"])["runs"]
    .transform("mean")
)

batting_match[["batter", "bowling_team", "opponent_avg_run"]].head()


,batter,bowling_team,opponent_avg_run
0,AA Noffke,Kolkata Knight Riders,9.000000
1,B Akhil,Kolkata Knight Riders,0.000000
2,BB McCullum,Royal Challengers Bangalore,59.000000
3,CL White,Kolkata Knight Riders,16.333333
4,DJ Hussey,Royal Challengers Bangalore,20.250000


In [13]:
bowling_match["opponent_avg_wkts"] = (
    bowling_match.groupby(["bowler", "batting_team"])["wickets"]
    .transform("mean")
)
bowling_match[["bowler", "batting_team", "opponent_avg_wkts"]].head()

,bowler,batting_team,opponent_avg_wkts
0,AA Noffke,Kolkata Knight Riders,1.000000
1,AB Agarkar,Royal Challengers Bangalore,1.000000
2,AB Dinda,Royal Challengers Bangalore,1.250000
3,CL White,Kolkata Knight Riders,0.000000
4,I Sharma,Royal Challengers Bangalore,0.714286


In [14]:
batting_match["career_avg_runs"] = (
    batting_match
    .groupby("batter")["runs"]
    .expanding()
    .mean()
    .reset_index(level=0, drop=True)
)

batting_match["career_strike_rate"] = (
    (batting_match["runs"] / batting_match["balls"]) * 100
)

batting_match[["batter", "career_avg_runs", "career_strike_rate"]].head()


,batter,career_avg_runs,career_strike_rate
0,AA Noffke,9.0,90.000000
1,B Akhil,0.0,0.000000
2,BB McCullum,158.0,216.438356
3,CL White,6.0,60.000000
4,DJ Hussey,12.0,100.000000


In [ ]:
bowling_match["career_avg_wkts"] = (
    bowling_match.groupby("bowler")["wickets"].transform("mean")
)

bowling_match[["bowler", "wickets", "career_avg_wkts"]].head(10)

,bowler,wickets,career_avg_wkts
0,AA Noffke,1.0,1.000000
1,AB Agarkar,3.0,0.700000
2,AB Dinda,2.0,0.714286
3,CL White,0.0,0.000000
4,I Sharma,1.0,0.827586
5,JH Kallis,1.0,0.466667
6,LR Shukla,1.0,0.687500
7,P Kumar,0.0,1.000000
8,SB Joshi,0.0,0.250000
9,SC Ganguly,2.0,0.571429


In [ ]:
batting_match["target_next_runs"] = (
    batting_match
    .groupby("batter")["runs"]
    .shift(-1)
)

batting_match[["batter", "runs", "target_next_runs"]].head()


,batter,runs,target_next_runs
0,AA Noffke,9.0,NaN
1,B Akhil,0.0,3.0
2,BB McCullum,158.0,5.0
3,CL White,6.0,31.0
4,DJ Hussey,12.0,38.0


In [ ]:
bowling_match["target_next_wkts"] = (
    bowling_match.groupby("bowler")["wickets"].shift(-1)
)
bowling_match[["bowler", "wickets", "target_next_wkts"]].head()

,bowler,wickets,target_next_wkts
0,AA Noffke,1.0,NaN
1,AB Agarkar,3.0,2.0
2,AB Dinda,2.0,1.0
3,CL White,0.0,NaN
4,I Sharma,1.0,1.0


In [ ]:
bat_match = batting_match.dropna()
bowl_match = bowling_match.dropna()

print("Final Bat Shape:", bat_match.shape)
bat_match.head()
print("Final Bowl shape:",bowl_match.shape)
bowl_match.head()

Final Bat Shape: (1164, 15)
Final Bowl shape: (1445, 15)


,match_id,date,bowler,bowling_team,batting_team,venue,wickets,runs_conceded,balls_bowled,overs,wkts_last_5,venue_avg_wkts,opponent_avg_wkts,career_avg_wkts,target_next_wkts
1,335982,2008-04-18,AB Agarkar,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,3.0,25.0,24.0,4.000000,3.0,3.000000,1.333333,0.700000,2.0
2,335982,2008-04-18,AB Dinda,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,2.0,9.0,18.0,3.000000,2.0,2.000000,1.000000,0.714286,1.0
4,335982,2008-04-18,I Sharma,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,1.0,7.0,18.0,3.000000,1.0,1.000000,0.800000,0.827586,1.0
5,335982,2008-04-18,JH Kallis,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,1.0,48.0,24.0,4.000000,1.0,0.555556,0.333333,0.466667,0.0
6,335982,2008-04-18,LR Shukla,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,1.0,12.0,7.0,1.166667,1.0,1.000000,0.500000,0.687500,0.0
